# 🚀 Trénování středního SLM modelu na Google Colab (Toki Pona)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fandap42/DP_SLM/blob/dev/experiments/notebooks/train_medium_colab.ipynb)

Tento notebook slouží k natrénování a kompletní evaluaci **středního jazykového modelu (Medium SLM, ~14,4M parametrů)** pro jazyk Toki Pona v rámci diplomové práce na Katedře statistiky a pravděpodobnosti FIS VŠE.

### 🏛️ Architektura a 4 pilíře hodnocení:
1. **Jazykový model:** Causal Decoder-Only Transformer s RMSNorm, GELU a akcelerovanou PyTorch Scaled Dot-Product Attention (SDPA).
   - Konfigurace `medium`: `d_model = 384`, `n_layer = 8`, `n_head = 8`, `d_ff = 1536`.
2. **1. Pilíř:** Normalizovaná křížová entropie (**BPC**) a doménový rozpad (*in-domain* Tatoeba vs. *out-of-domain* Wikipesija, Poki Lapo, Lipu Sewi).
3. **2. Pilíř:** Gramatická přijatelnost (**BLiMP Toki Pona** – 447 minimálních párů pokrývajících částice *li*, *e*, *pi*, *la*, slovosled a lexikální validitu).
4. **3. Pilíř:** Kompoziční zobecnění (**Hold-out Compounding** – testování 20 neviděných složenin vůči 20 známým spojením).
5. **4. Pilíř:** Statistická syntéza (ANOVA, LMM smíšené modely, GLMM logistická regrese a odhad empirického Kaplanova mocninného zákona).

## 1. Kontrola akcelerátoru (GPU / CUDA)

In [ ]:
!nvidia-smi
import torch
print("PyTorch verze:", torch.__version__)
print("CUDA k dispozici:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Zařízení:", torch.cuda.get_device_name(0))

## 2. Klonování repozitáře a instalace balíčku

In [ ]:
import os
if not os.path.exists("/content/DP_SLM"):
    !git clone -b dev https://github.com/fandap42/DP_SLM.git /content/DP_SLM
%cd /content/DP_SLM
!pip install -e .

## 3. Generování testovacích lingvistických sad

In [ ]:
!slm-generate-tests

## 4. Konfigurace tréninku pro střední model (`medium.json`)

In [ ]:
import json
from pathlib import Path

cfg_path = Path("experiments/configs/medium.json")
if not cfg_path.exists():
    cfg = {
        "experiment_name": "medium_run",
        "description": "Medium model (~14.4M params) on full Toki Pona dataset",
        "data_fractions": [1.00],
        "model_sizes": ["medium"],
        "tokenizers": ["word"],
        "seeds": [42],
        "epochs": 5,
        "batch_size": 32,
        "learning_rate": 0.0005,
        "mixed_precision": True
    }
    cfg_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cfg_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2)

print(cfg_path.read_text(encoding="utf-8"))

## 5. Spuštění tréninku a komplexní 4-pilířové evaluace

In [ ]:
!python experiments/scripts/run_experiments.py --config experiments/configs/medium.json

## 6. Vyhodnocení výsledků a přehled metrik

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv("results/metrics/results.csv")
cols = [
    "exp_id", "model_size", "tokenizer", "data_fraction", 
    "test_bpc", "bpc_in_domain", "bpc_out_domain", "domain_gap",
    "grammar_acc", "comp_pair_accuracy", "comp_generalization_gap"
]
available_cols = [c for c in cols if c in df.columns]
display(df[available_cols].tail(10))

## 7. Interaktivní generování textu (Demo)

In [ ]:
!slm-demo --run exp_001_d100_medium_word_s42 --prompt "jan pona mi li"